In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('WA_Marketing-Campaign.csv')

print(df.shape)
print(df.head())
print(df.isna().sum())
print('Точек продаж:', df['LocationID'].nunique())

# Каждая точка должна встречаться ровно 4 раза (4 недели)
print('Проверка:', (df.groupby('LocationID').size() == 4).all())

# Распределение промо
print(df['Promotion'].value_counts().sort_index())

(548, 7)
   MarketID MarketSize  LocationID  AgeOfStore  Promotion  week  \
0         1     Medium           1           4          3     1   
1         1     Medium           1           4          3     2   
2         1     Medium           1           4          3     3   
3         1     Medium           1           4          3     4   
4         1     Medium           2           5          2     1   

   SalesInThousands  
0             33.73  
1             35.67  
2             29.03  
3             39.25  
4             27.81  
MarketID            0
MarketSize          0
LocationID          0
AgeOfStore          0
Promotion           0
week                0
SalesInThousands    0
dtype: int64
Точек продаж: 137
Проверка: True
Promotion
1    172
2    188
3    188
Name: count, dtype: int64


In [4]:
promo_stats = df.groupby('Promotion')['SalesInThousands'].agg(
    ['count', 'mean', 'std']).round(2)

print(promo_stats)

           count   mean    std
Promotion                     
1            172  58.10  16.55
2            188  47.33  15.11
3            188  55.36  16.77


In [5]:
# Промо по размеру рынка: если в одной акции все рынки Small, а в другой Large — сравнение нечестное
print(pd.crosstab(df['Promotion'], df['MarketSize']))

# Средний возраст магазинов по акциям
print(df.groupby('Promotion')['AgeOfStore'].mean().round(1))

# Промо по рынкам
print(pd.crosstab(df['Promotion'], df['MarketID']))

MarketSize  Large  Medium  Small
Promotion                       
1              56      96     20
2              64     108     16
3              48     116     24
Promotion
1    8.3
2    8.0
3    9.2
Name: AgeOfStore, dtype: float64
MarketID   1   2   3   4   5   6   7   8   9   10
Promotion                                        
1          20   4  28  16   8  20  16  20  12  28
2          20   0  24  16  32  24  16   8   8  40
3          12  20  36   4  20  16  28  20  20  12


In [6]:
from scipy import stats

# 1. Основной тест по всем 548 наблюдениям
groups = [df[df['Promotion'] == p]['SalesInThousands'] for p in [1,2,3]]
f_stat, p_anova = stats.f_oneway(*groups)
print(f"Все наблюдения -> F = {f_stat:.2f}, p-value = {p_anova:.3e}")


Все наблюдения -> F = 21.95, p-value = 6.766e-10


In [7]:
# 2. Бонусный тест (усредняем 4 недели по каждой точке, чтобы убрать повторные измерения)
loc = df.groupby(['LocationID', 'Promotion'], as_index=False)['SalesInThousands'].mean()
groups_loc = [loc[loc['Promotion'] == p]['SalesInThousands'] for p in [1,2,3]]
f_stat_loc, p_anova_loc = stats.f_oneway(*groups_loc)
print(f"По точкам продаж -> F = {f_stat_loc:.2f}, p_value = {p_anova_loc:.3e}")

По точкам продаж -> F = 5.85, p_value = 3.681e-03


In [9]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(df['SalesInThousands'], df['Promotion'], alpha=0.05)
print(tukey)

tukey_df = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])

 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower    upper  reject
-----------------------------------------------------
     1      2 -10.7696    0.0 -14.7738 -6.7654   True
     1      3  -2.7345 0.2444  -6.7388  1.2697  False
     2      3   8.0351    0.0   4.1208 11.9493   True
-----------------------------------------------------


Раскатывать промо-2 не стоит — проигрыш доказан. Выбор между промо-1 и промо-3 статистикой не решается: решать будут стоимость кампании, механика и стратегия. Если промо-3 дешевле в запуске — брать его; если цены равны — есть смысл взять промо-1, у которого среднее выше, но без статистической гарантии.

In [12]:
anova = pd.DataFrame({
    'тест':['по наблюдениям', 'по точкам'],
    'F':[round(f_stat, 2), round(f_stat_loc, 2)],
    'p_value':[p_anova, p_anova_loc]
})

by_market = df.groupby(['MarketSize', 'Promotion'])['SalesInThousands'].mean().unstack().round(2)
by_marketid = df.groupby(['MarketID', 'Promotion'])['SalesInThousands'].mean().unstack().round(2)
by_week = df.groupby(['week', 'Promotion'])['SalesInThousands'].mean().unstack().round(2)

with pd.ExcelWriter('promo_report.xlsx') as writer:
    promo_stats.to_excel(writer, sheet_name='KPI')
    anova.to_excel(writer, sheet_name='ANOVA', index=False)
    tukey_df.to_excel(writer, sheet_name='Tukey', index=False)
    by_market.to_excel(writer, sheet_name='По рынкам')
    by_marketid.to_excel(writer, sheet_name='По Market ID')
    by_week.to_excel(writer, sheet_name='По неделям')
    df.to_excel(writer, sheet_name='Данные', index=False)